# E10 — Comparação ponta a ponta com o método original (Area1)

**Versão experimental independente.** Preserva o notebook original e a V2. Usa exclusivamente os arquivos já existentes em `Area1` e os checkpoints que a V2 eventualmente tenha gravado. Nenhuma linha desta cópia apresenta resultados novos antes de uma execução real.

**Ordem:** (1) configurar Colab com RAM alta, (2) executar Quickshift **global** e as etapas originais, (3) executar E10 para oito classificadores e avaliação pareada por referências comuns. O fluxo da etapa (2) vem das células históricas da V2; gráficos decorativos foram omitidos. Diferentes versões das bibliotecas ou colisões de IDs podem impedir reproduzir exatamente os valores históricos.

**Atenção:** o Quickshift global pode consumir dezenas de GiB e falhar no Colab. A execução V2 por quadrantes não é substituta do original. A etapa E10 só informa métricas calculadas quando a execução concluir.

In [ ]:
%pip -q install rasterio geopandas shapely scikit-image scikit-learn scipy joblib psutil
from google.colab import drive
drive.mount('/content/drive')
import os, json, gc, warnings, psutil
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, rasterio, shapely
from rasterio import features
from rasterio.mask import mask
from skimage.segmentation import quickshift
from skimage.feature import canny
from skimage.color import rgb2hsv
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

AREA_E10=Path('/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1')
OUT_E10=Path('/content/drive/MyDrive/Projetos/Segmentacao_Breno/outputs/e10_original_vs_v2')
TILES_E10=Path('/content/drive/MyDrive/Projetos/Segmentacao_Breno/outputs/v2_progressivo_256/tiles')
for name in ('AOI_RGB_NEW.tif','AOI_NEW_MULTI.tif','AOI_new.shp','Arvores.gpkg','Grama.gpkg'):
    if not (AREA_E10/name).is_file():raise FileNotFoundError(AREA_E10/name)
with rasterio.open(AREA_E10/'AOI_RGB_NEW.tif') as rr:
    print('Raster original:',rr.width,'x',rr.height,'pixel(s) | CRS',rr.crs)
print('RAM livre GiB:',round(psutil.virtual_memory().available/1024**3,2))
print('Recomendação: Colab com alta RAM; espaço livre e tempo de execução não são garantidos.')

## A. Pipeline global — reprodução rastreável do método histórico

As células seguintes reproduzem as operações originais de leitura, Quickshift global, vetorização, atributos RGB/NIR, K-Means de seis grupos, seleção do cluster 3 e atribuição dos rótulos de referência. **Não execute esta parte com 4 GiB de memória:** há risco de encerramento da sessão.

A conversão histórica dos identificadores de segmentos para `uint16` é mantida somente se não houver estouro. Se houver identificadores acima de 65535, será utilizado `int32` com aviso explícito: isso **corrige colisões** e deixa de ser uma réplica literal do arquivo histórico.

In [ ]:
path_img = '/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1/AOI_RGB_NEW.tif'
src_rgb = rasterio.open(path_img)
im = src_rgb.read()
im  = im.transpose([1,2,0])
'''R = src_rgb.read(3)
G = src_rgb.read(2)
B = src_rgb.read(1)


R = R / 10000
G = G / 10000
B = B / 10000

R = R * 255
G = G * 255
B = B * 255

R = R.astype('uint8')
G = G.astype('uint8')
B = B.astype('uint8')

im = np.dstack((R,G,B))'''
print('RGB global carregado:',im.shape,im.dtype)

In [ ]:
AOI = gpd.read_file('/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1/AOI_new.shp')
print('AOI carregada:',len(AOI),'geometria(s)')

In [ ]:
segments_quick2 = quickshift(im, kernel_size=3, max_dist=5, ratio=0.04)
print('Quickshift global:',segments_quick2.shape,'máximo ID:',int(segments_quick2.max()))

In [ ]:
if int(segments_quick2.max())>np.iinfo(np.uint16).max:
    if int(segments_quick2.max())>np.iinfo(np.int32).max:
        raise OverflowError('Mais identificadores do que cabem em int32.')
    MODO_SEGMENTOS_E10='int32_corrigido: original uint16 causaria colisao'
    warnings.warn('IDs acima de 65535: o cast uint16 do histórico colidiria. Usando int32; não é reprodução bit a bit.')
    segments_quick2=segments_quick2.astype(np.int32)
else:
    MODO_SEGMENTOS_E10='uint16_historico_sem_colisao'
    segments_quick2=segments_quick2.astype(np.uint16)
print('Representação de IDs:',MODO_SEGMENTOS_E10)

In [ ]:
from rasterio import features
from shapely.geometry import shape
import geopandas as gpd
import shapely
import json
from rasterio.mask import mask
import pandas as pd
find_shapes = features.shapes(segments_quick2, transform=src_rgb.transform)
polygons = [shapely.geometry.Polygon(shape[0]["coordinates"][0]) for shape in find_shapes]
crs = src_rgb.crs
Poly_gdf = gpd.GeoDataFrame(crs=crs, geometry=polygons)
print('Polígonos globais antes do recorte:',len(Poly_gdf))

In [ ]:
clip_poly = Poly_gdf.clip(AOI)
clip_poly = clip_poly.reset_index(drop=True)
json_dataset = json.loads(clip_poly.to_json())['features']
print('Polígonos após recorte pela AOI:',len(clip_poly))

In [ ]:
path_img_mult = '/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1/AOI_NEW_MULTI.tif'
src = rasterio.open(path_img_mult)

X = []
Y = []
for i in range(len(json_dataset)):
  coords = json_dataset[i]['geometry']
  out_image, out_transform = mask(src, [coords], crop=True, nodata=0)

  out_image = out_image.transpose([1,2,0])

  R_mean = out_image[:,:,0][np.nonzero(out_image[:,:,0])].mean()
  G_mean = out_image[:,:,1][np.nonzero(out_image[:,:,1])].mean()
  B_mean = out_image[:,:,2][np.nonzero(out_image[:,:,2])].mean()
  N_mean = out_image[:,:,3][np.nonzero(out_image[:,:,3])].mean()
  R_std = out_image[:,:,0][np.nonzero(out_image[:,:,0])].std()
  G_std = out_image[:,:,1][np.nonzero(out_image[:,:,1])].std()
  B_std = out_image[:,:,2][np.nonzero(out_image[:,:,2])].std()
  N_std = out_image[:,:,3][np.nonzero(out_image[:,:,3])].std()




  X.append([R_mean,G_mean,B_mean,N_mean,R_std,G_std,B_std,N_std])
X = np.array(X)
df = pd.DataFrame(X, columns=['R_Mean','G_Mean','B_Mean','N_Mean','R_Std', 'G_Std', 'B_Std', 'N_Std'])
df.dropna(inplace=True)
print('Objetos com oito atributos válidos:',len(df))

In [ ]:
from sklearn.cluster import KMeans

# Apply K-Means clustering
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10) # You can change the number of clusters
kmeans_clusters = kmeans.fit_predict(df[['R_Mean','G_Mean','B_Mean','N_Mean','R_Std', 'G_Std', 'B_Std', 'N_Std']])

# Add K-Means cluster labels to the DataFrame
df['kmeans_cluster'] = kmeans_clusters

display(df.head())
clip_poly['kmeans_cluster'] = df['kmeans_cluster']
print('Candidatos cluster 3:',int((clip_poly.kmeans_cluster==3).sum()))

In [ ]:

path_grama = '/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1/Grama.gpkg'
path_arvore = '/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1/Arvores.gpkg'
gdf_grama = gpd.read_file(path_grama)
gdf_arvore = gpd.read_file(path_arvore)
gdf_grama['Classe'] = 'grama'
gdf_arvore['Classe'] = 'arvore'
gdf_full = pd.concat([gdf_grama, gdf_arvore])
gdf_full.to_crs(clip_poly.crs, inplace=True)
gdf_arvore_seg = clip_poly[clip_poly['kmeans_cluster'] == 3]
gdf_arvore_seg_filter = gdf_arvore_seg.sjoin(gdf_full, how='inner', predicate='intersects')
print('Pares objeto–referência após cluster 3:',len(gdf_arvore_seg_filter))

## B. Três atributos originais por segmento

A ordem abaixo é a histórica: NDVI, bordas Canny e brilho HSV a partir do raster multiespectral. A extração final usa máscara de polígonos do cluster 3 e grava apenas um TIFF intermediário de atributos na sessão. Os gráficos originais foram removidos desta cópia para economizar RAM, sem alterar as fórmulas.

In [ ]:
src = rasterio.open(path_img_mult)
img = src.read()
nir = img[3]
red = img[0]
ndvi=(nir.astype(float)-red.astype(float))/(nir.astype(float)+red.astype(float)+1e-8)
nir_uint=nir/10000
edges=canny(nir_uint,sigma=1)
img_rgb=img[0:3].transpose([1,2,0])
img_hsv=rgb2hsv(img_rgb)
intensity=img_hsv[:,:,2]
print('Mapas globais calculados:',ndvi.shape,edges.shape,intensity.shape)

In [ ]:
# Convert edges (boolean) to float for stacking
edges_float = edges.astype(np.float32)

# Stack intensity and edges as two separate bands
# The order here is intensity first, then edges
stacked_bands = np.stack([intensity, edges_float, ndvi], axis=0)

# Get metadata from the original source (src) for writing the new TIFF
profile = src.profile

# Update profile for the new stacked image
# Change the number of bands to 2 and data type to float32
profile.update(
    dtype=rasterio.float32,
    count=3,
    nodata=None
)

# Define the output path for the new TIFF file
output_filepath = 'features_stack.tif'

# Write the stacked image to a new TIFF file
with rasterio.open(output_filepath, 'w', **profile) as dst:
    dst.write(stacked_bands)

print('Raster intermediário de atributos:',output_filepath)

In [ ]:
src = rasterio.open(output_filepath)
json_dataset = json.loads(gdf_arvore_seg_filter.to_json())['features']

X = []
for i in range(len(json_dataset)):
  coords = json_dataset[i]['geometry']
  out_image, out_transform = mask(src, [coords], crop=True, nodata=-999)
  classe_values = json_dataset[i]['properties']['Classe']

  mean_intensity = out_image[0][out_image[0] != -999].mean()
  mean_edges = out_image[1][out_image[1] != -999].mean()
  mean_ndvi = out_image[2][out_image[2] != -999].mean()
  X.append([mean_intensity, mean_edges, mean_ndvi,classe_values])

df = pd.DataFrame(X, columns=['mean_intensity','mean_edges','mean_ndvi','classe'])
print('Tabela histórica pronta:',df.shape,'rótulos:',df.classe.value_counts().to_dict())
assert len(df)==len(gdf_arvore_seg_filter)
print('Pronto para rodar a etapa E10.')

## C. E10 — oito classificadores, partições comuns e cobertura

Este arquivo de código lê o objeto `df` e a junção espacial produzidos acima, audita as referências, executa os **oito classificadores** na divisão histórica 80/20, audita segmentos únicos e, quando existem checkpoints V2, utiliza a interseção de referências para testar os **dois pipelines nas mesmas divisões**. O embargo pareado de 25 m é medido **entre pontos**, não entre contornos completos; por isso não é idêntico ao E3c.

Todos os resultados efetivamente calculados (CSV e manifesto JSON) ficam na pasta `outputs/e10_original_vs_v2` do mesmo Drive. Se não houver checkpoints V2, a comparação pareada é indicada como incompleta; nenhum resultado é inventado.

In [ ]:
from urllib.request import urlretrieve
E10_SCRIPT='/content/e10_original_vs_v2.py'
urlretrieve('https://raw.githubusercontent.com/BrenoBragaGalvao/mestrado-arvores-urbanas/main/experimentos/e10_original_vs_v2.py',E10_SCRIPT)
%run -i /content/e10_original_vs_v2.py

## Interpretação e limites

- **Histórico:** usar a primeira tabela para comparar os oito modelos sob as mesmas associações e o mesmo sorteio 80/20 histórico. Ela pode conter segmentos duplicados.
- **Original deduplicado:** população diferente, sem reaproveitar segmentos entre treino/teste.
- **Original × V2 pareados:** usar **somente referências presentes em ambos**, com os mesmos índices de treino e teste, e medir também a cobertura; a seleção global do cluster 3 pode deixar muitas referências fora.
- **Atenção científica:** se ocorrer correção de overflow para `int32`, o resultado reproduz o algoritmo global e o K-Means, mas não o cast histórico para `uint16`; registrar isso na dissertação.
- A V2 de quadrantes **não é** a execução integral comprovada de todas as células da V2; aqui o braço V2 utiliza apenas checkpoints efetivamente existentes.
- Não declarar risco elétrico real, altura verdadeira, teste externo entre cidades nem superioridade estatística sem evidências.